# DataJam 2026 · ConviveData  
## Cuaderno 01 · Limpieza de fuentes originales desde GitHub

Este cuaderno documenta el primer tramo del flujo reproducible del proyecto: la limpieza de las fuentes originales. Su propósito es tomar los archivos crudos alojados en el repositorio, estandarizar campos mínimos, filtrar el periodo 2018-2025, conservar únicamente las 20 localidades de Bogotá y exportar bases limpias que puedan ser utilizadas por el cuaderno de ejecución.

El flujo evita rutas locales como `C:\Users\...` y permite que el cuaderno corra desde Google Colab. Para lograrlo, primero busca los archivos en la estructura local del repositorio y, si no existen, los descarga desde GitHub Raw.

Fuentes procesadas:

- Violencia intrafamiliar.
- Lesiones personales.
- Consumo abusivo o problemático de SPA.

Reglas generales:

- Periodo de análisis: 2018 a 2025.
- Unidad territorial: localidad.
- Solo se conservan registros localizados en las 20 localidades de Bogotá.
- Los registros sin localidad, sin dato o no aplicables se excluyen de la base localizada.
- Las fuentes no se cruzan a nivel de caso; se preparan para integración territorial posterior.

## 1. Librerías

Se importan las librerías necesarias para lectura de archivos, manejo de rutas, normalización de textos y procesamiento tabular.

- `pandas` y `numpy` permiten manipular las bases.
- `Path` permite usar rutas de forma compatible entre Windows, Linux y Colab.
- `urllib.request` permite descargar archivos desde GitHub Raw si no existen localmente.
- `unicodedata` y `re` se usan para limpiar nombres de localidades y evitar errores por tildes, espacios o variantes de escritura.

In [ ]:
from pathlib import Path
import re
import unicodedata
import urllib.request
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. Configuración reproducible de rutas

Esta sección define las carpetas de trabajo del proyecto y los enlaces públicos a las fuentes alojadas en GitHub.

La lógica es la siguiente:

1. Se define la URL base del repositorio en formato GitHub Raw.
2. Se crean las carpetas `data/raw`, `outputs_limpieza` y `data/processed` si no existen.
3. Se construye un diccionario con la ruta local esperada y la URL remota de cada fuente.
4. La función `descargar_si_no_existe()` usa el archivo local si ya está disponible; si no, lo descarga automáticamente.

Esto hace que el cuaderno pueda ejecutarse tanto localmente como en Colab sin que el usuario cambie rutas manualmente.

In [ ]:
# Repositorio público en GitHub Raw.
# Estos enlaces permiten ejecutar el cuaderno desde Colab sin rutas locales.
REPO_RAW = "https://raw.githubusercontent.com/00Jake00/convivedata-vif-bogota-2018-2025-Datajam/main"

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "outputs_limpieza"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVOS_RAW = {
    "vif": {
        "local": RAW_DIR / "violencia_intrafamiliar.csv",
        "url": f"{REPO_RAW}/data/raw/violencia_intrafamiliar.csv",
    },
    "lesiones": {
        "local": RAW_DIR / "lesiones_personales.csv",
        "url": f"{REPO_RAW}/data/raw/lesiones_personales.csv",
    },
    "spa": {
        "local": RAW_DIR / "consumo_abusivo_spa.xlsx",
        "url": f"{REPO_RAW}/data/raw/consumo_abusivo_spa.xlsx",
    },
}

def descargar_si_no_existe(nombre):
    """Usa el archivo local si existe; si no existe, lo descarga desde GitHub Raw."""
    ruta = ARCHIVOS_RAW[nombre]["local"]
    url = ARCHIVOS_RAW[nombre]["url"]

    if not ruta.exists():
        print(f"Descargando {nombre} desde GitHub Raw...")
        urllib.request.urlretrieve(url, ruta)
    else:
        print(f"{nombre}: archivo local encontrado")

    return ruta

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 3. Descarga o localización de fuentes crudas

En esta celda se ejecuta la descarga o localización de los tres archivos originales.

Si el cuaderno se ejecuta en Colab, normalmente la carpeta `data/raw` no existe al iniciar. Por eso, la función descarga los archivos desde GitHub Raw y los guarda en la estructura esperada.

Si el cuaderno se ejecuta localmente dentro del repositorio, simplemente usa los archivos existentes.

In [ ]:
ruta_vif = descargar_si_no_existe("vif")
ruta_lesiones = descargar_si_no_existe("lesiones")
ruta_spa = descargar_si_no_existe("spa")

print("Rutas de entrada:")
print("VIF:", ruta_vif)
print("Lesiones:", ruta_lesiones)
print("SPA:", ruta_spa)

## 4. Funciones de lectura y normalización

Esta sección concentra las funciones de apoyo para limpiar los datos.

La normalización territorial es fundamental porque los nombres de localidades pueden aparecer con diferencias de tildes, mayúsculas, espacios o variantes. Por ejemplo, `Los Mártires`, `LOS MARTIRES` y `Mártires` deben terminar con una misma llave territorial.

Las funciones principales son:

- `leer_csv_seguro()`: intenta leer archivos CSV con diferentes codificaciones comunes.
- `quitar_tildes()`: elimina tildes para crear llaves homogéneas.
- `normalizar_localidad()`: estandariza los nombres de localidad.
- `marcar_localizacion()`: identifica si una fila pertenece a una de las 20 localidades válidas de Bogotá.

Esta etapa evita que errores de escritura fragmenten una misma localidad en varias categorías.

In [ ]:
def leer_csv_seguro(ruta, sep=";"):
    """
    Lee CSV probando varias codificaciones comunes.
    En estas fuentes el separador esperado es punto y coma.
    """
    for encoding in ["utf-8-sig", "utf-8", "latin1", "cp1252"]:
        try:
            return pd.read_csv(ruta, sep=sep, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer el archivo: {ruta}")


def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


ALIAS_LOCALIDAD = {
    "CANDELARIA": "LA CANDELARIA",
    "LA CANDELARIA": "LA CANDELARIA",
    "MARTIRES": "LOS MARTIRES",
    "LOS MARTIRES": "LOS MARTIRES",
    "RAFAEL URIBE": "RAFAEL URIBE URIBE",
    "RAFAEL URIBE URIBE": "RAFAEL URIBE URIBE",
    "CIUDAD BOLIVAR": "CIUDAD BOLIVAR",
    "SAN CRISTOBAL": "SAN CRISTOBAL",
    "USAQUEN": "USAQUEN",
    "FONTIBON": "FONTIBON",
    "ENGATIVA": "ENGATIVA",
    "BOGOTA D.C.": "BOGOTA D.C.",
    "BOGOTA DC": "BOGOTA D.C.",
}

LOCALIDADES_VALIDAS = [
    "USAQUEN", "CHAPINERO", "SANTA FE", "SAN CRISTOBAL", "USME",
    "TUNJUELITO", "BOSA", "KENNEDY", "FONTIBON", "ENGATIVA",
    "SUBA", "BARRIOS UNIDOS", "TEUSAQUILLO", "LOS MARTIRES",
    "ANTONIO NARINO", "PUENTE ARANDA", "LA CANDELARIA",
    "RAFAEL URIBE URIBE", "CIUDAD BOLIVAR", "SUMAPAZ"
]

LOCALIDAD_DISPLAY = {
    "USAQUEN": "Usaquén",
    "CHAPINERO": "Chapinero",
    "SANTA FE": "Santa Fe",
    "SAN CRISTOBAL": "San Cristóbal",
    "USME": "Usme",
    "TUNJUELITO": "Tunjuelito",
    "BOSA": "Bosa",
    "KENNEDY": "Kennedy",
    "FONTIBON": "Fontibón",
    "ENGATIVA": "Engativá",
    "SUBA": "Suba",
    "BARRIOS UNIDOS": "Barrios Unidos",
    "TEUSAQUILLO": "Teusaquillo",
    "LOS MARTIRES": "Los Mártires",
    "ANTONIO NARINO": "Antonio Nariño",
    "PUENTE ARANDA": "Puente Aranda",
    "LA CANDELARIA": "La Candelaria",
    "RAFAEL URIBE URIBE": "Rafael Uribe Uribe",
    "CIUDAD BOLIVAR": "Ciudad Bolívar",
    "SUMAPAZ": "Sumapaz",
}

def normalizar_localidad(valor):
    if pd.isna(valor):
        return np.nan
    texto = quitar_tildes(valor).upper().strip()
    texto = re.sub(r"\s+", " ", texto)
    return ALIAS_LOCALIDAD.get(texto, texto)

def marcar_localizacion(df, col="loc_norm"):
    base = df.copy()
    base["localidad_valida"] = base[col].isin(LOCALIDADES_VALIDAS)
    base["localidad_display"] = base[col].map(LOCALIDAD_DISPLAY)
    return base

## 5. Carga de fuentes originales

Aquí se leen las fuentes crudas en memoria.

Las dos fuentes en CSV se leen con separador `;`, que es frecuente en archivos exportados con configuración regional en español. La fuente de consumo abusivo de SPA se lee desde Excel.

El resumen de carga permite verificar el número de filas y columnas originales antes de aplicar filtros. Este control es útil para dejar trazabilidad sobre el punto de partida del procesamiento.

In [ ]:
vif_raw = leer_csv_seguro(ruta_vif, sep=";")
lesiones_raw = leer_csv_seguro(ruta_lesiones, sep=";")
spa_raw = pd.read_excel(ruta_spa, sheet_name="in")

resumen_carga = pd.DataFrame([
    {"fuente": "Violencia intrafamiliar", "filas_originales": len(vif_raw), "columnas_originales": len(vif_raw.columns)},
    {"fuente": "Lesiones personales", "filas_originales": len(lesiones_raw), "columnas_originales": len(lesiones_raw.columns)},
    {"fuente": "Consumo abusivo SPA", "filas_originales": len(spa_raw), "columnas_originales": len(spa_raw.columns)},
])

resumen_carga

## 6. Limpieza de VIF y lesiones personales

Violencia intrafamiliar y lesiones personales tienen una estructura similar. Por eso se usa una función común para procesarlas.

La función realiza estos pasos:

1. Estandariza nombres de columnas.
2. Verifica que existan las columnas requeridas.
3. Convierte año, mes y cantidad a formato numérico.
4. Estandariza variables categóricas como sexo, hecho, rango vital, rango del día y arma empleada.
5. Normaliza la localidad.
6. Marca si la fila está localizada en una de las 20 localidades válidas.
7. Filtra el periodo 2018-2025.
8. Filtra el hecho esperado de cada fuente.
9. Separa base localizada y base sin localización.

La base localizada es la que entra al análisis territorial. La base sin localización queda como control de calidad, no como insumo del visor.

In [ ]:
def limpiar_base_seguridad(df, hecho_esperado, nombre_fuente):
    base = df.copy()
    base.columns = [str(c).strip() for c in base.columns]

    columnas_requeridas = [
        "SEXO", "ANIO", "MES", "HECHO", "LOCALIDAD", "COD_LOCALIDAD",
        "ARMA_EMPLEADA", "RANGO_VITAL", "RANGO_DEL_DIA", "CANTIDAD"
    ]
    faltantes = [c for c in columnas_requeridas if c not in base.columns]
    if faltantes:
        raise ValueError(f"{nombre_fuente}: faltan columnas requeridas: {faltantes}")

    base["ANIO"] = pd.to_numeric(base["ANIO"], errors="coerce")
    base["MES"] = pd.to_numeric(base["MES"], errors="coerce")
    base["CANTIDAD"] = pd.to_numeric(base["CANTIDAD"], errors="coerce").fillna(0)

    for col in ["HECHO", "SEXO", "RANGO_VITAL", "RANGO_DEL_DIA", "ARMA_EMPLEADA"]:
        base[col] = base[col].astype(str).str.upper().str.strip()

    base["loc_norm"] = base["LOCALIDAD"].apply(normalizar_localidad)
    base = marcar_localizacion(base)

    base = base[base["ANIO"].between(2018, 2025)].copy()
    base = base[base["HECHO"] == hecho_esperado].copy()
    base["fuente"] = nombre_fuente

    columnas = [
        "fuente", "SEXO", "ANIO", "MES", "HECHO", "LOCALIDAD", "loc_norm",
        "localidad_display", "localidad_valida", "COD_LOCALIDAD", "ARMA_EMPLEADA",
        "RANGO_VITAL", "RANGO_DEL_DIA", "CANTIDAD"
    ]
    return base[columnas]


vif_limpia = limpiar_base_seguridad(
    vif_raw,
    hecho_esperado="VIOLENCIA INTRAFAMILIAR",
    nombre_fuente="Violencia intrafamiliar"
)

lesiones_limpia = limpiar_base_seguridad(
    lesiones_raw,
    hecho_esperado="LESIONES PERSONALES",
    nombre_fuente="Lesiones personales"
)

vif_localizada = vif_limpia[vif_limpia["localidad_valida"]].copy()
vif_sin_localizacion = vif_limpia[~vif_limpia["localidad_valida"]].copy()

lesiones_localizada = lesiones_limpia[lesiones_limpia["localidad_valida"]].copy()
lesiones_sin_localizacion = lesiones_limpia[~lesiones_limpia["localidad_valida"]].copy()

print("VIF limpia:", vif_limpia.shape)
print("VIF localizada:", vif_localizada.shape)
print("Casos VIF localizados:", int(vif_localizada["CANTIDAD"].sum()))
print()
print("Lesiones limpia:", lesiones_limpia.shape)
print("Lesiones localizada:", lesiones_localizada.shape)
print("Casos lesiones localizados:", int(lesiones_localizada["CANTIDAD"].sum()))

## 7. Limpieza de consumo abusivo de SPA

La base de consumo abusivo de SPA tiene una estructura diferente, por lo que se procesa aparte.

En esta etapa se verifican columnas mínimas, se convierten año, mes y número de casos a formato numérico, se normaliza la localidad de residencia y se filtra el periodo 2018-2025.

La variable de conteo usada es `CASOS`. Al igual que en las demás fuentes, se conserva una base localizada y una base no localizada para control.

In [ ]:
spa_limpia = spa_raw.copy()
spa_limpia.columns = [str(c).strip() for c in spa_limpia.columns]

columnas_requeridas_spa = ["ANO", "MESNOTIFICACION", "NOMBRELOCALIDADRESIDENCIA", "CASOS"]
faltantes_spa = [c for c in columnas_requeridas_spa if c not in spa_limpia.columns]
if faltantes_spa:
    raise ValueError(f"Consumo abusivo SPA: faltan columnas requeridas: {faltantes_spa}")

spa_limpia["ANO"] = pd.to_numeric(spa_limpia["ANO"], errors="coerce")
spa_limpia["MESNOTIFICACION"] = pd.to_numeric(spa_limpia["MESNOTIFICACION"], errors="coerce")
spa_limpia["CASOS"] = pd.to_numeric(spa_limpia["CASOS"], errors="coerce").fillna(0)

spa_limpia["loc_norm"] = spa_limpia["NOMBRELOCALIDADRESIDENCIA"].apply(normalizar_localidad)
spa_limpia = marcar_localizacion(spa_limpia)
spa_limpia["fuente"] = "Consumo abusivo SPA"

spa_limpia = spa_limpia[spa_limpia["ANO"].between(2018, 2025)].copy()

columnas_spa = [
    "fuente", "ANO", "MESNOTIFICACION", "SEXO", "NOMBRELOCALIDADRESIDENCIA",
    "loc_norm", "localidad_display", "localidad_valida", "CURSO_DE_VIDA",
    "TIPOASEGURAMIENTO", "NIVELEDUCATIVO", "NOMBREUPZ", "CASOS"
]
columnas_spa = [c for c in columnas_spa if c in spa_limpia.columns]
spa_limpia = spa_limpia[columnas_spa]

spa_localizada = spa_limpia[spa_limpia["localidad_valida"]].copy()
spa_sin_localizacion = spa_limpia[~spa_limpia["localidad_valida"]].copy()

print("SPA limpia:", spa_limpia.shape)
print("SPA localizada:", spa_localizada.shape)
print("Casos SPA localizados:", int(spa_localizada["CASOS"].sum()))

## 8. Resumen de control

El resumen de control consolida los resultados de la limpieza por fuente.

Incluye:

- registros limpios;
- registros localizados;
- registros no localizados;
- casos localizados;
- periodo observado;
- indicación de si la fuente entra al visor.

Este bloque permite verificar que la limpieza produjo los totales esperados antes de continuar con el análisis. En particular, el total localizado de VIF para 2018-2025 debe coincidir con el valor usado en el visor territorial.

In [ ]:
resumen_limpieza = pd.DataFrame([
    {
        "fuente": "Violencia intrafamiliar",
        "registros_limpios": len(vif_limpia),
        "registros_localizados": len(vif_localizada),
        "registros_no_localizados": len(vif_sin_localizacion),
        "casos_localizados": int(vif_localizada["CANTIDAD"].sum()),
        "periodo": f"{int(vif_limpia['ANIO'].min())}-{int(vif_limpia['ANIO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
    {
        "fuente": "Lesiones personales",
        "registros_limpios": len(lesiones_limpia),
        "registros_localizados": len(lesiones_localizada),
        "registros_no_localizados": len(lesiones_sin_localizacion),
        "casos_localizados": int(lesiones_localizada["CANTIDAD"].sum()),
        "periodo": f"{int(lesiones_limpia['ANIO'].min())}-{int(lesiones_limpia['ANIO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
    {
        "fuente": "Consumo abusivo SPA",
        "registros_limpios": len(spa_limpia),
        "registros_localizados": len(spa_localizada),
        "registros_no_localizados": len(spa_sin_localizacion),
        "casos_localizados": int(spa_localizada["CASOS"].sum()),
        "periodo": f"{int(spa_limpia['ANO'].min())}-{int(spa_limpia['ANO'].max())}",
        "entra_al_visor_actual": "Sí",
    },
])

resumen_limpieza

## 9. Exportación de bases limpias

Las bases limpias se exportan en dos ubicaciones:

- `outputs_limpieza/`: carpeta de salidas generadas automáticamente por el cuaderno.
- `data/processed/`: carpeta de productos procesados/finales del repositorio.

Esto permite que el flujo sea reproducible y, al mismo tiempo, que los resultados limpios queden disponibles para revisión directa en GitHub.

In [ ]:
salidas = {
    "vif_limpia_localizada_2018_2025.csv": vif_localizada,
    "lesiones_limpia_localizada_2018_2025.csv": lesiones_localizada,
    "spa_abusivo_limpio_localizado_2018_2025.csv": spa_localizada,
    "resumen_limpieza_fuentes.csv": resumen_limpieza,
}

for nombre, df in salidas.items():
    df.to_csv(OUTPUT_DIR / nombre, index=False, encoding="utf-8-sig")
    df.to_csv(PROCESSED_DIR / nombre, index=False, encoding="utf-8-sig")

print("Archivos exportados en:", OUTPUT_DIR)
for archivo in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", archivo.name)

print("\nCopia adicional en:", PROCESSED_DIR)

## 10. Cierre del cuaderno de limpieza

Al terminar esta etapa, quedan disponibles las bases limpias requeridas por el cuaderno de ejecución:

- `vif_limpia_localizada_2018_2025.csv`
- `lesiones_limpia_localizada_2018_2025.csv`
- `spa_abusivo_limpio_localizado_2018_2025.csv`
- `resumen_limpieza_fuentes.csv`

El siguiente cuaderno toma estas bases y construye indicadores, rankings, pruebas estadísticas exploratorias y productos finales para el visor.